In [1]:
import joblib
import numpy as np
import pandas as pd
from typing import List, Dict

In [2]:
MODEL_PATH = 'models/risk_model.pkl'
model = joblib.load(MODEL_PATH)
FEATURES = ['weekly_work_hours', 'overtime_hours', 'meeting_density',
            'consecutive_work_days', 'conflict_count', 'workload_trend', 'night_weekend_hours']

def generate_recommendations(features: Dict) -> List[str]:
    recs = []
    if features['overtime_hours'] > 6:
        recs.append("Высокий овертайм: делегируйте вечерние задачи, добавьте буфер 1ч/день")
    if features['meeting_density'] > 0.65:
        recs.append("Meetings >65%: внедрите 'день без встреч', сократите длительность до 30 мин")
    if features['consecutive_work_days'] >= 6:
        recs.append("6+ дней подряд: запланируйте выходной, перераспределите нагрузку")
    if features['conflict_count'] >= 3:
        recs.append("⚠Частые коллизии: синхронизируйте календари, заблокируйте фокус-время")
    if features['workload_trend'] > 0.25:
        recs.append("Нагрузка растёт: снизьте входящие задачи на 15-20% на след. спринт")
    if not recs:
        recs.append("Риск низкий. График сбалансирован.")
    return recs

In [3]:
def predict_risk(features: Dict) -> Dict:
    X = pd.DataFrame([features])[FEATURES]
    score = float(np.clip(model.predict(X)[0], 0, 10))

    trend = features.get('workload_trend', 0)
    forecast = [round(np.clip(score + trend*(i+1), 0, 10), 1) for i in range(3)]

    category = "low" if score < 4 else "medium" if score < 7 else "high"
    recs = generate_recommendations(features)

    return {
        "score": round(score, 1),
        "category": category,
        "forecast_3d": forecast,
        "recommendations": recs,
        "feature_importance": dict(zip(FEATURES, model.feature_importances_.round(3)))
    }